# 第五部分：回归分析

本 notebook 对应第五部分任务，包含：

- **5.1 CAPM 模型估计**：对 10 只股票分别估计 CAPM，汇总 $\hat{\alpha}$、p 值、$\hat{\beta}$、95% 置信区间、$R^2$，并绘制 Beta 系数图。
- **5.2 宏观指标对股票收益率的影响（选做）**：选择 1 项宏观指标，以月度数据回归 10 只股票的月度收益率，汇报 $\hat{\gamma}$ 及显著性，并讨论行业差异。

运行前请确认你已经完成前面各部分，并且项目根目录下至少已有这些文件：

- `data/clean/stock_clean.csv`
- `data/stock/stock_universe.csv`
- `data/index/index_000300.csv`
- `data/macro/macro_lpr_1y.csv` 或 `data/macro/macro_cpi_yoy.csv`

本 notebook 会把结果保存到 `output/` 目录。

In [ ]:
# 如缺少依赖，会自动安装
import sys
import subprocess
import importlib

required = ["pandas", "numpy", "matplotlib", "statsmodels", "scipy"]
for pkg in required:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

print("依赖检查完成。")

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import statsmodels.api as sm
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
CLEAN_DIR = DATA_DIR / "clean"
STOCK_DIR = DATA_DIR / "stock"
INDEX_DIR = DATA_DIR / "index"
MACRO_DIR = DATA_DIR / "macro"
OUTPUT_DIR = ROOT / "output"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

print("项目根目录：", ROOT)
print("输出目录：", OUTPUT_DIR)

## 读取并整理基础数据

下面这一步会读取个股清洗后数据、股票清单、沪深300指数数据，以及宏观数据。  
其中宏观部分优先使用 **1年期 LPR**，若该文件不存在，则自动退回使用 **CPI 同比增速**。

In [ ]:
stock_universe_path = STOCK_DIR / "stock_universe.csv"
if stock_universe_path.exists():
    meta = pd.read_csv(stock_universe_path, dtype={"code": str})
else:
    raise FileNotFoundError(f"未找到 {stock_universe_path}")

meta["code"] = meta["code"].astype(str).str.zfill(6)
meta = meta[["code", "name", "industry"]].drop_duplicates()

stock_clean_path = CLEAN_DIR / "stock_clean.csv"
if not stock_clean_path.exists():
    raise FileNotFoundError(f"未找到 {stock_clean_path}")

stocks = pd.read_csv(stock_clean_path, dtype={"code": str})
stocks["code"] = stocks["code"].astype(str).str.zfill(6)

drop_cols = [c for c in stocks.columns if str(c).lower().startswith("unnamed")]
if drop_cols:
    stocks = stocks.drop(columns=drop_cols)

if "date" not in stocks.columns:
    raise ValueError("stock_clean.csv 缺少 date 列")
stocks["date"] = pd.to_datetime(stocks["date"])

for col in ["open", "high", "low", "close", "volume", "amount"]:
    if col in stocks.columns:
        stocks[col] = pd.to_numeric(stocks[col], errors="coerce")

required_cols = ["date", "code", "close"]
for col in required_cols:
    if col not in stocks.columns:
        raise ValueError(f"stock_clean.csv 缺少必要列：{col}")

stocks = stocks.sort_values(["code", "date"]).reset_index(drop=True)

if "log_return" not in stocks.columns:
    stocks["log_return"] = stocks.groupby("code")["close"].transform(lambda s: np.log(s / s.shift(1)))

stocks = stocks.merge(meta, on="code", how="left")

hs300_path = INDEX_DIR / "index_000300.csv"
if not hs300_path.exists():
    raise FileNotFoundError(f"未找到 {hs300_path}")

hs300 = pd.read_csv(hs300_path)
drop_cols = [c for c in hs300.columns if str(c).lower().startswith("unnamed")]
if drop_cols:
    hs300 = hs300.drop(columns=drop_cols)

hs300["date"] = pd.to_datetime(hs300["date"])
for col in ["open", "high", "low", "close", "volume", "amount"]:
    if col in hs300.columns:
        hs300[col] = pd.to_numeric(hs300[col], errors="coerce")
hs300 = hs300.sort_values("date").reset_index(drop=True)
hs300["market_log_return"] = np.log(hs300["close"] / hs300["close"].shift(1))

macro_lpr = MACRO_DIR / "macro_lpr_1y.csv"
macro_cpi = MACRO_DIR / "macro_cpi_yoy.csv"

if macro_lpr.exists():
    macro_path = macro_lpr
    macro_name = "1年期LPR"
elif macro_cpi.exists():
    macro_path = macro_cpi
    macro_name = "CPI同比增速"
else:
    raise FileNotFoundError("未找到宏观数据文件：macro_lpr_1y.csv 或 macro_cpi_yoy.csv")

macro = pd.read_csv(macro_path)
drop_cols = [c for c in macro.columns if str(c).lower().startswith("unnamed")]
if drop_cols:
    macro = macro.drop(columns=drop_cols)

if "date" not in macro.columns or "value" not in macro.columns:
    raise ValueError(f"{macro_path.name} 需要至少包含 date 和 value 两列")

macro["date"] = pd.to_datetime(macro["date"])
macro["value"] = pd.to_numeric(macro["value"], errors="coerce")
macro = macro.dropna(subset=["date", "value"]).sort_values("date").reset_index(drop=True)

print("股票数据：", stocks.shape)
print("沪深300数据：", hs300.shape)
print(f"宏观数据（{macro_name}）：", macro.shape)

display(stocks.head())
display(hs300.head())
display(macro.head())

## 5.1 CAPM 模型估计

模型设定为：

\[
r_{i,t} - r_f = \alpha_i + \beta_i (r_{m,t} - r_f) + \varepsilon_{i,t}
\]

其中：

- $r_{i,t}$：个股日对数收益率  
- $r_{m,t}$：沪深300日对数收益率  
- $r_f$：无风险利率，统一设为年化 2.0%，日频换算为 $0.02 / 252$

下方代码会对 10 只股票分别估计 OLS，并输出回归汇总表。

In [ ]:
rf_daily = 0.02 / 252

capm_rows = []

for code, grp in stocks.groupby("code"):
    temp = grp[["date", "code", "name", "industry", "log_return"]].merge(
        hs300[["date", "market_log_return"]],
        on="date",
        how="inner"
    ).dropna()

    if len(temp) < 30:
        continue

    y = temp["log_return"] - rf_daily
    x = temp["market_log_return"] - rf_daily
    X = sm.add_constant(x)

    model = sm.OLS(y, X).fit()

    alpha = model.params["const"]
    beta = model.params["market_log_return"]
    alpha_p = model.pvalues["const"]
    beta_ci_low, beta_ci_high = model.conf_int().loc["market_log_return"].tolist()
    r2 = model.rsquared

    capm_rows.append({
        "code": code,
        "stock": temp["name"].iloc[0] if "name" in temp.columns else code,
        "industry": temp["industry"].iloc[0] if "industry" in temp.columns else "",
        "alpha_hat": alpha,
        "alpha_p_value": alpha_p,
        "beta_hat": beta,
        "beta_ci_low": beta_ci_low,
        "beta_ci_high": beta_ci_high,
        "r_squared": r2,
        "n_obs": len(temp)
    })

capm_results = pd.DataFrame(capm_rows).sort_values(["industry", "beta_hat"], ascending=[True, False]).reset_index(drop=True)
capm_results["95% CI"] = capm_results.apply(
    lambda r: f"[{r['beta_ci_low']:.3f}, {r['beta_ci_high']:.3f}]",
    axis=1
)

capm_table = capm_results[[
    "stock", "industry", "alpha_hat", "alpha_p_value", "beta_hat", "95% CI", "r_squared"
]].copy()

capm_table = capm_table.rename(columns={
    "stock": "股票",
    "industry": "行业",
    "alpha_hat": "α̂",
    "alpha_p_value": "p值",
    "beta_hat": "β̂",
    "r_squared": "R²"
})

capm_results.to_csv(OUTPUT_DIR / "capm_results.csv", index=False, encoding="utf-8-sig")
capm_table.to_csv(OUTPUT_DIR / "capm_results_table.csv", index=False, encoding="utf-8-sig")

display(capm_table.style.format({
    "α̂": "{:.6f}",
    "p值": "{:.4f}",
    "β̂": "{:.3f}",
    "R²": "{:.3f}"
}))

print("CAPM 结果已保存：")
print("-", OUTPUT_DIR / "capm_results.csv")
print("-", OUTPUT_DIR / "capm_results_table.csv")

In [ ]:
plot_df = capm_results.sort_values(["industry", "beta_hat"], ascending=[True, False]).reset_index(drop=True)

industries = list(plot_df["industry"].fillna("未知行业").unique())
cmap = plt.get_cmap("tab10")
color_map = {ind: cmap(i % 10) for i, ind in enumerate(industries)}
colors = [color_map[ind] for ind in plot_df["industry"].fillna("未知行业")]

y_pos = np.arange(len(plot_df))
beta = plot_df["beta_hat"].values
err_left = beta - plot_df["beta_ci_low"].values
err_right = plot_df["beta_ci_high"].values - beta

fig, ax = plt.subplots(figsize=(10, 7))
ax.errorbar(beta, y_pos, xerr=[err_left, err_right], fmt="none", ecolor="gray", capsize=4, alpha=0.8)
ax.scatter(beta, y_pos, c=colors, s=80)
ax.axvline(1.0, linestyle="--", linewidth=1.5, color="black")

ax.set_yticks(y_pos)
ax.set_yticklabels(plot_df["stock"])
ax.invert_yaxis()
ax.set_xlabel("Beta 值")
ax.set_ylabel("股票名称")
ax.set_title("CAPM Beta 系数图（含95%置信区间）")

legend_handles = [
    Line2D([0], [0], marker="o", color="w", label=ind, markerfacecolor=color_map[ind], markersize=8)
    for ind in industries
]
ax.legend(handles=legend_handles, title="行业", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_capm_beta.png", dpi=200, bbox_inches="tight")
plt.show()

### CAPM 结果解读

下面的文字不是只列数字，而是根据回归结果自动总结：

1. 哪些股票的 $\hat{\beta}>1$，它们集中在哪些行业；  
2. $\hat{\alpha}$ 是否显著异于零；  
3. 哪些股票的 $R^2$ 最高和最低，以及这意味着什么。

In [ ]:
high_beta = capm_results[capm_results["beta_hat"] > 1].copy()
sig_alpha = capm_results[capm_results["alpha_p_value"] < 0.05].copy()
top_r2 = capm_results.loc[capm_results["r_squared"].idxmax()]
low_r2 = capm_results.loc[capm_results["r_squared"].idxmin()]

if len(high_beta) > 0:
    hb_text = "、".join([f"{r.stock}（{r.industry}，β={r.beta_hat:.2f}）" for r in high_beta.itertuples()])
else:
    hb_text = "样本中没有 β 明显大于 1 的股票"

if len(sig_alpha) > 0:
    sa_text = "、".join([f"{r.stock}（p={r.alpha_p_value:.4f}）" for r in sig_alpha.itertuples()])
    alpha_comment = (
        f"从 Alpha 显著性看，样本中显著异于零的股票包括：{sa_text}。"
        "这意味着这些股票在控制市场超额收益以后，仍可能存在无法被单因子 CAPM 完全解释的异常收益。"
    )
else:
    alpha_comment = (
        "从 Alpha 显著性看，样本中的 α 在 5% 水平下大多并不显著异于零。"
        "这说明就该样本而言，单因子 CAPM 对日收益率的解释已经覆盖了主要的系统性市场风险溢价。"
    )

discussion = f'''
**问题 1：哪些股票的 β̂ > 1？它们属于哪些行业？这是否与“周期性 vs 防御性”分类吻合？**  
β̂ 大于 1 的股票有：{hb_text}。这些股票通常对市场整体波动更敏感，若主要集中在汽车、能源、通讯等行业，则与“周期性行业风险暴露更高”的经验判断基本一致；相对而言，银行或白酒中的部分个股若 β 较低，则更接近稳健或防御性特征。

**问题 2：α̂ 是否显著异于零？Alpha 显著意味着什么？**  
{alpha_comment}

**问题 3：R² 最高和最低的股票分别是哪只？你如何解释这一差异？**  
样本中 $R^2$ 最高的是 **{top_r2['stock']}**（{top_r2['industry']}，$R^2={top_r2['r_squared']:.3f}$），最低的是 **{low_r2['stock']}**（{low_r2['industry']}，$R^2={low_r2['r_squared']:.3f}$）。  
这说明前者的收益波动与市场因子的同步性更强，市场收益可以解释其更大比例的日度波动；而后者可能受公司特质、行业政策、事件冲击或经营基本面变化的影响更大，因此单因子 CAPM 的解释力相对较弱。
'''
display(Markdown(discussion))

## 5.2 宏观指标对股票收益率的影响（选做）

这里选择 **一项宏观指标**（优先 1年期 LPR，否则使用 CPI 同比增速），以月度数据分析其对 10 只股票月度收益率的影响：

\[
r^{月}_{i,t} = \alpha_i + \gamma_i \cdot X_t + \varepsilon_{i,t}
\]

其中：

- $r^{月}_{i,t}$：个股月度对数收益率  
- $X_t$：选定的宏观指标  
- $\gamma_i$：股票对该宏观变量的敏感度

注意：该部分是选做，我已一并写入，便于你直接提交完整结果。

In [ ]:
monthly_price = (
    stocks[["date", "code", "name", "industry", "close"]]
    .dropna()
    .sort_values(["code", "date"])
    .set_index("date")
    .groupby("code")
    .resample("M")["close"]
    .last()
    .reset_index()
)

monthly_price = monthly_price.merge(meta, on="code", how="left", suffixes=("", "_meta"))
if "name_meta" in monthly_price.columns:
    monthly_price["name"] = monthly_price["name"].fillna(monthly_price["name_meta"])
if "industry_meta" in monthly_price.columns:
    monthly_price["industry"] = monthly_price["industry"].fillna(monthly_price["industry_meta"])

monthly_price["monthly_log_return"] = monthly_price.groupby("code")["close"].transform(lambda s: np.log(s / s.shift(1)))
monthly_price["month"] = monthly_price["date"].dt.to_period("M").astype(str)

macro_monthly = macro.copy()
macro_monthly["month"] = macro_monthly["date"].dt.to_period("M").astype(str)
macro_monthly = macro_monthly.groupby("month", as_index=False)["value"].last()
macro_monthly = macro_monthly.rename(columns={"value": "macro_value"})

monthly_panel = monthly_price.merge(macro_monthly, on="month", how="left")
monthly_panel = monthly_panel.dropna(subset=["monthly_log_return", "macro_value"]).reset_index(drop=True)

print("月度回归样本：", monthly_panel.shape)
display(monthly_panel.head())

In [ ]:
macro_rows = []

for code, grp in monthly_panel.groupby("code"):
    temp = grp[["code", "name", "industry", "monthly_log_return", "macro_value"]].dropna()
    if len(temp) < 8:
        continue

    y = temp["monthly_log_return"]
    X = sm.add_constant(temp["macro_value"])
    model = sm.OLS(y, X).fit()

    gamma = model.params["macro_value"]
    gamma_p = model.pvalues["macro_value"]
    ci_low, ci_high = model.conf_int().loc["macro_value"].tolist()

    macro_rows.append({
        "code": code,
        "stock": temp["name"].iloc[0],
        "industry": temp["industry"].iloc[0],
        "macro_indicator": macro_name,
        "gamma_hat": gamma,
        "gamma_p_value": gamma_p,
        "gamma_ci_low": ci_low,
        "gamma_ci_high": ci_high,
        "r_squared": model.rsquared,
        "n_obs": len(temp)
    })

macro_results = pd.DataFrame(macro_rows).sort_values(["industry", "gamma_hat"], ascending=[True, False]).reset_index(drop=True)
macro_results["95% CI"] = macro_results.apply(
    lambda r: f"[{r['gamma_ci_low']:.3f}, {r['gamma_ci_high']:.3f}]",
    axis=1
)

macro_table = macro_results[[
    "stock", "industry", "macro_indicator", "gamma_hat", "gamma_p_value", "95% CI", "r_squared"
]].copy()

macro_table = macro_table.rename(columns={
    "stock": "股票",
    "industry": "行业",
    "macro_indicator": "宏观指标",
    "gamma_hat": "γ̂",
    "gamma_p_value": "p值",
    "r_squared": "R²"
})

macro_results.to_csv(OUTPUT_DIR / "macro_monthly_regression_results.csv", index=False, encoding="utf-8-sig")
macro_table.to_csv(OUTPUT_DIR / "macro_monthly_regression_table.csv", index=False, encoding="utf-8-sig")

display(macro_table.style.format({
    "γ̂": "{:.4f}",
    "p值": "{:.4f}",
    "R²": "{:.3f}"
}))
print("宏观月度回归结果已保存：")
print("-", OUTPUT_DIR / "macro_monthly_regression_results.csv")
print("-", OUTPUT_DIR / "macro_monthly_regression_table.csv")

In [ ]:
plot_df = macro_results.sort_values(["industry", "gamma_hat"], ascending=[True, False]).reset_index(drop=True)

industries = list(plot_df["industry"].fillna("未知行业").unique())
cmap = plt.get_cmap("tab10")
color_map = {ind: cmap(i % 10) for i, ind in enumerate(industries)}
colors = [color_map[ind] for ind in plot_df["industry"].fillna("未知行业")]

y_pos = np.arange(len(plot_df))
gamma = plot_df["gamma_hat"].values
err_left = gamma - plot_df["gamma_ci_low"].values
err_right = plot_df["gamma_ci_high"].values - gamma

fig, ax = plt.subplots(figsize=(10, 7))
ax.errorbar(gamma, y_pos, xerr=[err_left, err_right], fmt="none", ecolor="gray", capsize=4, alpha=0.8)
ax.scatter(gamma, y_pos, c=colors, s=80)
ax.axvline(0.0, linestyle="--", linewidth=1.5, color="black")

ax.set_yticks(y_pos)
ax.set_yticklabels(plot_df["stock"])
ax.invert_yaxis()
ax.set_xlabel(f"{macro_name} 的 γ 值")
ax.set_ylabel("股票名称")
ax.set_title(f"{macro_name} 对个股月收益率影响的系数图（含95%置信区间）")

legend_handles = [
    Line2D([0], [0], marker="o", color="w", label=ind, markerfacecolor=color_map[ind], markersize=8)
    for ind in industries
]
ax.legend(handles=legend_handles, title="行业", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_macro_gamma.png", dpi=200, bbox_inches="tight")
plt.show()

### 宏观敏感性结果解读

下面会根据估计结果自动总结：

1. 哪些股票对所选宏观指标更敏感；  
2. 不同行业的敏感度差异；  
3. 背后的经济逻辑。

In [ ]:
macro_results["abs_gamma"] = macro_results["gamma_hat"].abs()

top_sensitive = macro_results.sort_values("abs_gamma", ascending=False).head(3)
industry_mean = (
    macro_results.groupby("industry", as_index=False)["abs_gamma"]
    .mean()
    .sort_values("abs_gamma", ascending=False)
)

top_text = "、".join([f"{r.stock}（{r.industry}，γ={r.gamma_hat:.4f}）" for r in top_sensitive.itertuples()])
top_industry = industry_mean.iloc[0]["industry"]
low_industry = industry_mean.iloc[-1]["industry"]

macro_discussion = f'''
**宏观指标选择：{macro_name}**  
本部分选择的宏观指标是 **{macro_name}**，它与企业融资成本、估值贴现率、消费能力或风险偏好存在联系，因此可能通过不同渠道影响股票收益率。

**哪些股票更敏感？**  
从估计结果看，对该指标最敏感的股票主要包括：{top_text}。这些股票对应的回归系数绝对值更大，说明其月度收益率对宏观变量变化的响应更明显。

**行业差异如何理解？**  
按行业平均绝对敏感度比较，敏感度相对更高的行业是 **{top_industry}**，而相对较低的行业是 **{low_industry}**。  
如果所选指标是利率类变量（如 LPR），则高杠杆、资本开支大或估值依赖贴现率的行业往往更敏感；如果是通胀类变量（如 CPI），则成本传导能力、终端需求稳定性和定价权强弱会导致不同行业的响应方向和强度出现差异。
'''
display(Markdown(macro_discussion))

## 文件输出说明

运行完本 notebook 后，你可以在 `output/` 目录检查以下结果文件：

- `capm_results.csv`
- `capm_results_table.csv`
- `fig_capm_beta.png`
- `macro_monthly_regression_results.csv`
- `macro_monthly_regression_table.csv`
- `fig_macro_gamma.png`

若老师要求文件名统一，你可以把本 notebook 重命名为 `04_regression.ipynb` 或直接保留当前名称。

In [ ]:
print("第五部分全部完成。你现在可以检查 output/ 目录中的文件：")
for p in [
    OUTPUT_DIR / "capm_results.csv",
    OUTPUT_DIR / "capm_results_table.csv",
    OUTPUT_DIR / "fig_capm_beta.png",
    OUTPUT_DIR / "macro_monthly_regression_results.csv",
    OUTPUT_DIR / "macro_monthly_regression_table.csv",
    OUTPUT_DIR / "fig_macro_gamma.png",
]:
    print("-", p)